# EDA — AAVE V3 WETH utilization

Quick look at:
- the utilization series itself (level, ΔU, autocorrelation structure),
- the relationship with realized vol and DVOL,
- the marginal distribution of every engineered feature,
- and the empirical class balance at the chosen 7-day horizon.

Run order:
1. `python -m data.fetch_subgraph`
2. `python -m data.fetch_ohlcv && python -m data.fetch_funding && python -m data.fetch_options && python -m data.fetch_gas`
3. open this notebook.

In [ ]:
import sys
from pathlib import Path

# Allow imports from the repo root when the notebook is opened from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf

import config
from features.builders import build_dataset, load_inputs

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 140)

## Raw inputs

In [ ]:
ins = load_inputs()
weth = ins.weth
print(f"WETH reserve: {len(weth):,} daily rows from {weth.index[0].date()} → {weth.index[-1].date()}")
weth.head()

## Utilization timeseries

The horizontal line is `optimalUsageRatio` (the "kink"). Most regimes you'll see
U pulled toward this point — that's the rate model doing its job.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(weth.index, weth["U"], lw=1.0)
axes[0].axhline(weth["opt_U"].iloc[-1], ls="--", color="grey", label=f"optimalU={weth['opt_U'].iloc[-1]:.2f}")
axes[0].set_ylabel("Utilization U")
axes[0].legend(loc="upper left")

axes[1].plot(weth.index, weth["U"].diff(), lw=0.7)
axes[1].axhline(0, color="grey", ls="--")
axes[1].set_ylabel("ΔU (1d)")

axes[2].plot(weth.index, weth["liquidity_rate"] * 100, label="supply APR", lw=1.0)
axes[2].plot(weth.index, weth["borrow_rate"] * 100, label="borrow APR", lw=1.0)
axes[2].set_ylabel("APR (%)")
axes[2].legend(loc="upper left")

fig.suptitle("AAVE V3 WETH — utilization & rates", fontsize=12)
plt.tight_layout()

## Autocorrelation structure

If `U` has long-memory autocorrelation but `ΔU` does not, the level is mean-reverting
around a slow drift. That's the regime where our half-life and gap-to-kink features
do most of the work. If `ΔU` has positive lag-1 autocorrelation, the model can also
exploit short-term momentum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(weth["U"].dropna(), lags=30, ax=axes[0], title="ACF — U (level)")
plot_acf(weth["U"].diff().dropna(), lags=30, ax=axes[1], title="ACF — ΔU")
plt.tight_layout()

## Feature matrix

Build the full `(X, y)` dataset and inspect the class balance and feature
distributions. If `y.mean()` is far from 0.5 the `class_weight="balanced"`
in the model matters.

In [ ]:
X, y, meta = build_dataset()
print(f"X shape: {X.shape}")
print(f"y class balance (P[U_t+7 > U_t]): {y.mean():.3f}")
print(f"date range: {X.index[0].date()} → {X.index[-1].date()}")
X.describe().T[["mean", "std", "min", "50%", "max"]].round(3)

In [ ]:
corr = X.corr()
fig, ax = plt.subplots(figsize=(10, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=False, ax=ax, cbar_kws={"shrink": 0.6})
ax.set_title("Feature correlation matrix")
plt.tight_layout()

## Univariate predictive power

Quick non-parametric check: for each feature, what's the AUC for predicting
"U up over the next 7 days"? Anything noticeably above 0.55 is meaningful;
anything around 0.50 is noise. Use this to spot features that aren't pulling
their weight.

In [ ]:
from sklearn.metrics import roc_auc_score

aucs = []
for col in X.columns:
    s = X[col]
    if s.std() == 0:
        continue
    auc = roc_auc_score(y, s.fillna(s.median()))
    # AUC < 0.5 means the feature is anti-predictive at its raw sign — flip for display
    aucs.append((col, max(auc, 1 - auc), auc))

univ = pd.DataFrame(aucs, columns=["feature", "auc_oriented", "auc_raw"]).sort_values("auc_oriented", ascending=False)
univ.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(4, len(univ) * 0.25)))
top = univ.head(20)[::-1]
ax.barh(top["feature"], top["auc_oriented"] - 0.5)
ax.axvline(0, color="grey")
ax.set_xlabel("Univariate AUC − 0.5 (oriented)")
ax.set_title("Top features by univariate AUC")
plt.tight_layout()